### 1. Run YOLO object detection on streaming videos detection (Just demo to show the performance of YOLO)

In [9]:
from pathlib import Path
import cv2
from ultralytics import YOLO

# =========================
# CONFIG
# =========================
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    # Jupyter notebook fallback
    BASE_DIR = Path.cwd().parent
    
WEIGHTS_PATH = BASE_DIR / "yolo_weight" / "violence-yolov8n-v3.pt" 
# VIDEO_DIR = BASE_DIR / "data" / "raw" / "Real Life Violence Dataset" / "NonViolence"   # folder that contains videos
# VIDEO_DIR = 'd:\BaiduNetdiskDownload'
VIDEO_DIR = BASE_DIR / "data" / "streaming videos" / "raw_samples"/ "The_Boys_S4_E3_(30.00-33.00)_sample.mp4"
# VIDEO_DIR = BASE_DIR / "data" / "streaming videos" / "filter_results"
CONF_THRES = 0.25
IOU_THRES = 0.7
IMG_SIZE = 640

# =========================
# LOAD MODEL
# =========================
model = YOLO(WEIGHTS_PATH)

video_dir = Path(VIDEO_DIR)
video_exts = [".mp4", ".avi", ".mov", ".mkv"]

if video_dir.is_file():
    video_paths = [video_dir]
elif video_dir.is_dir():
    video_paths = sorted(
        [p for p in video_dir.iterdir() if p.suffix.lower() in video_exts]
    )
else:
    raise FileNotFoundError(f"Invalid path: {video_dir}")

if not video_paths:
    raise FileNotFoundError(f"No video files found in: {video_dir.resolve()}")

paused = False

for vp in video_paths:
    cap = cv2.VideoCapture(str(vp))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # fallback if fps is broken
    if fps <= 0:
        fps = 25

    delay = int(1000 / fps)

    if not cap.isOpened():
        print(f"[WARN] Could not open video: {vp}")
        continue

    window_name = f"YOLOv8 Detection - {vp.name}"
    print(f"\n[INFO] Playing: {vp}")

    while True:
        if not paused:
            ret, frame = cap.read()
            if not ret:
                break  # end of this video

            # YOLO inference on this frame
            results = model.predict(
                source=frame,
                imgsz=IMG_SIZE,
                conf=CONF_THRES,
                iou=IOU_THRES,
                verbose=False,
                classes=[1, 2, 3],  # when running on violence-yolov8n-v0, please delete this line
                device=0, 
            )

            # Draw boxes (Ultralytics built-in plot)
            annotated = results[0].plot()  # returns BGR image, ready for cv2.imshow

            cv2.imshow(window_name, annotated)

        key = cv2.waitKey(1) & 0xFF

        # Controls
        if key == ord("q"):          # quit all
            cap.release()
            cv2.destroyAllWindows()
            raise SystemExit
        elif key == ord("n"):        # next video
            break
        elif key == ord(" "):        # pause/resume
            paused = not paused

    cap.release()
    cv2.destroyWindow(window_name)

cv2.destroyAllWindows()
print("\n[INFO] Done.")



[INFO] Playing: d:\Myworkplace\Python\violence-movies\data\streaming videos\raw_samples\The_Boys_S4_E3_(30.00-33.00)_sample.mp4

[INFO] Done.


### 2. Using the trained YOLO model to filter violent clips

In [ ]:
# import os
# from pathlib import Path
# import cv2
# from ultralytics import YOLO

# try:
#     BASE_DIR = Path(__file__).resolve().parent.parent
# except NameError:
#     # Jupyter notebook fallback
#     BASE_DIR = Path.cwd().parent

# WEIGHTS_PATH = BASE_DIR / "yolo_weight" / "violence-yolov8n-v3.pt"

# # Input/output video file path
# VIDEO_DIR = BASE_DIR / "data" / "streaming videos" / "raw_samples" / "Hidden_Man_(1.55.00-2.03.00)_sample.mp4"
# OUTPUT_DIR = BASE_DIR / "data" / "streaming videos" / "filter_results"
# CONF_THRES = 0.3
# IOU_THRES = 0.7
# IMG_SIZE = 640
# K = 12 # same video when k consecutive frames detected

# # Load the trained YOLOv8 model
# model = YOLO(WEIGHTS_PATH)

# # Create the output folder (if it doesn't exist)
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Get all video files
# video_dir = Path(VIDEO_DIR)
# video_exts = [".mp4", ".avi", ".mov", ".mkv"]

# if video_dir.is_file():
#     video_paths = [video_dir]
# elif video_dir.is_dir():
#     video_paths = sorted(
#         [p for p in video_dir.iterdir() if p.suffix.lower() in video_exts]
#     )
# else:
#     raise FileNotFoundError(f"Invalid path: {video_dir}")

# if not video_paths:
#     raise FileNotFoundError(f"No video files found in: {video_dir.resolve()}")

# for video_file in video_paths:
#     cap = cv2.VideoCapture(str(video_file))
#     fps = cap.get(cv2.CAP_PROP_FPS)
#     frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

#     frame_count = 0
#     clip_number = 1
#     clip_frames = []
#     consec_det = 0
#     start_time = None
#     last_detection_time = None
#     clip_end_time = None

#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Perform object detection on the current frame
#         results = model.predict(
#             source=frame,
#             device=0,  # GPU 0
#             imgsz=IMG_SIZE,
#             conf=CONF_THRES,
#             iou=IOU_THRES,
#             classes=[1, 2, 3], # when running on violence-yolov8n-v0, please delete this line
#             verbose=False
#         )

#         # Extract detection results
#         detections = results[0].boxes.data.cpu().numpy()
#         current_time = frame_count / fps
#         if len(detections) > 0:
#             # If an object is detected, save the current frame
#             consec_det +=1
#             clip_frames.append(frame)
#             clip_end_time = current_time
#             if start_time is None:
#                 start_time = current_time
#             last_detection_time = current_time
#         else:
#             if consec_det < K:
#                 consec_det = 0
#                 clip_frames = []
#                 start_time = None
#                 last_detection_time = None
#                 frame_count += 1
#                 continue

#             # If no object is detected but the time interval is less than 5 seconds, save the current frame
#             if last_detection_time is not None and (current_time - last_detection_time < 5):
#                 clip_frames.append(frame)
#                 clip_end_time = current_time
#             else:
#                 # If the time interval is greater than 2 seconds and there are accumulated frames, save these frames as a video clip
#                 if len(clip_frames) > 0:
#                     end_time = clip_end_time
#                     start_time_formatted = f"{int(start_time // 60):02d}_{int(start_time % 60):02d}"
#                     end_time_formatted = f"{int(end_time // 60):02d}_{int(end_time % 60):02d}"
#                     clip_output_path = os.path.join(
#                         OUTPUT_DIR,
#                         f"{os.path.splitext(video_file)[0]}_clip_{clip_number}_{start_time_formatted}_to_{end_time_formatted}.mp4",
#                     )
#                     out = cv2.VideoWriter(
#                         clip_output_path,
#                         cv2.VideoWriter_fourcc(*"mp4v"),
#                         fps,
#                         (frame_width, frame_height),
#                     )
#                     for clip_frame in clip_frames:
#                         out.write(clip_frame)
#                     out.release()
#                     clip_frames = []
#                     clip_number += 1
#                     start_time = None
#                     last_detection_time = None
#                     consec_det = 0

#         frame_count += 1

#     # Process the remaining frames at the end of the video
#     if len(clip_frames) > 0:
#         end_time = clip_end_time
#         start_time_formatted = f"{int(start_time // 60):02d}_{int(start_time % 60):02d}"
#         end_time_formatted = f"{int(end_time // 60):02d}_{int(end_time % 60):02d}"
#         clip_output_path = os.path.join(
#             OUTPUT_DIR,
#             f"{os.path.splitext(video_file)[0]}_clip_{clip_number}_{start_time_formatted}_to_{end_time_formatted}.mp4",
#         )
#         out = cv2.VideoWriter(
#             clip_output_path,
#             cv2.VideoWriter_fourcc(*"mp4v"),
#             fps,
#             (frame_width, frame_height),
#         )
#         for clip_frame in clip_frames:
#             out.write(clip_frame)
#         out.release()

#     cap.release()

# # cv2.destroyAllWindows()

KeyboardInterrupt: 

In [8]:
import os
from pathlib import Path
import cv2
from ultralytics import YOLO

try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().parent

WEIGHTS_PATH = BASE_DIR / "yolo_weight" / "violence-yolov8n-v3.pt"

# Input / output
VIDEO_INPUT = BASE_DIR / "data" / "streaming videos" / "raw_samples" / "Game_of_Thrones_S1_E9_(33.00-35.00)_sample.mp4"
OUTPUT_DIR = BASE_DIR / "data" / "streaming videos" / "filter_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONF_THRES = 0.25
IOU_THRES = 0.7
IMG_SIZE = 640
K = 12
GAP_SECONDS = 5

# Speed control:
# 1 = check every frame (most accurate, slowest)
# 2 = check every 2nd frame (faster)
# 3 = check every 3rd frame (even faster)
FRAME_STRIDE = 1

model = YOLO(WEIGHTS_PATH)

video_exts = {".mp4", ".avi", ".mov", ".mkv"}
video_input = Path(VIDEO_INPUT)

if video_input.is_file():
    video_paths = [video_input]
elif video_input.is_dir():
    video_paths = sorted([p for p in video_input.iterdir() if p.suffix.lower() in video_exts])
else:
    raise FileNotFoundError(f"Invalid path: {video_input}")

if not video_paths:
    raise FileNotFoundError(f"No video files found in: {video_input}")


def format_time_mm_ss(t):
    m = int(t // 60)
    s = int(t % 60)
    return f"{m:02d}_{s:02d}"


for video_file in video_paths:
    cap = cv2.VideoCapture(str(video_file))   # FIX 1: open the real full path
    if not cap.isOpened():
        print(f"Cannot open video: {video_file}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30.0   # fallback

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    frame_count = 0
    clip_number = 1

    consec_det = 0
    start_time = None
    last_detection_time = None
    clip_end_time = None

    # Buffer only the first K detected frames before clip is officially opened
    pre_clip_frames = []

    writer = None
    temp_clip_path = None

    # For FRAME_STRIDE > 1, reuse the last prediction result on skipped frames
    last_has_det = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        current_time = frame_count / fps

        # Optional speed-up: do not run YOLO on every frame
        if frame_count % FRAME_STRIDE == 0:
            results = model.predict(
                source=frame,
                device=0,
                imgsz=IMG_SIZE,
                conf=CONF_THRES,
                iou=IOU_THRES,
                classes=[1, 2, 3],
                verbose=False
            )

            # FIX 2: do not force GPU->CPU numpy copy just to check empty/non-empty
            boxes = results[0].boxes
            last_has_det = boxes is not None and len(boxes) > 0

        has_det = last_has_det

        if has_det:
            consec_det += 1
            clip_end_time = current_time
            last_detection_time = current_time

            if start_time is None:
                start_time = current_time

            if writer is None:
                # still in warm-up stage before reaching K consecutive detections
                pre_clip_frames.append(frame)

                if consec_det >= K:
                    temp_clip_path = OUTPUT_DIR / f"{video_file.stem}_clip_{clip_number}_temp.mp4"
                    writer = cv2.VideoWriter(
                        str(temp_clip_path),
                        fourcc,
                        fps,
                        (frame_width, frame_height)
                    )

                    if not writer.isOpened():
                        raise RuntimeError(f"Cannot create output video: {temp_clip_path}")

                    # write the buffered first K frames
                    for f in pre_clip_frames:
                        writer.write(f)
                    pre_clip_frames.clear()
            else:
                # clip already confirmed, write directly
                writer.write(frame)

        else:
            if writer is None:
                # clip not confirmed yet
                if consec_det < K:
                    consec_det = 0
                    pre_clip_frames.clear()
                    start_time = None
                    last_detection_time = None
                    clip_end_time = None
            else:
                # clip already open: allow short gap with no detections
                if last_detection_time is not None and (current_time - last_detection_time) < GAP_SECONDS:
                    writer.write(frame)
                    clip_end_time = current_time
                else:
                    # close current clip
                    writer.release()

                    final_name = (
                        f"{video_file.stem}_clip_{clip_number}_"
                        f"{format_time_mm_ss(start_time)}_to_{format_time_mm_ss(clip_end_time)}.mp4"
                    )
                    final_clip_path = OUTPUT_DIR / final_name
                    os.replace(temp_clip_path, final_clip_path)

                    print(f"Saved: {final_clip_path}")

                    writer = None
                    temp_clip_path = None
                    pre_clip_frames.clear()

                    clip_number += 1
                    consec_det = 0
                    start_time = None
                    last_detection_time = None
                    clip_end_time = None

        frame_count += 1

    # finalize last clip at end of video
    if writer is not None:
        writer.release()

        final_name = (
            f"{video_file.stem}_clip_{clip_number}_"
            f"{format_time_mm_ss(start_time)}_to_{format_time_mm_ss(clip_end_time)}.mp4"
        )
        final_clip_path = OUTPUT_DIR / final_name
        os.replace(temp_clip_path, final_clip_path)

        print(f"Saved: {final_clip_path}")

    cap.release()

Saved: d:\Myworkplace\Python\violence-movies\data\streaming videos\filter_results\Game_of_Thrones_S1_E9_(33.00-35.00)_sample_clip_1_00_14_to_01_33.mp4
Saved: d:\Myworkplace\Python\violence-movies\data\streaming videos\filter_results\Game_of_Thrones_S1_E9_(33.00-35.00)_sample_clip_2_01_35_to_01_49.mp4
